# Data Onboarding and Contracts

## Situation

Riverside's evidence is spread across six systems and fourteen frozen sample records. The samples include stale policy, broken PDF layout, duplicate pages, a deleted autosave, missing rights territory, an API field rename, and a disabled contractor preserved in an old group snapshot. A document is therefore text plus ownership, access, version, lifecycle, and deletion behavior.

## Sketch

```mermaid
flowchart LR
    A["PDF, text, ERP, and API records"] --> B["Check owner and purpose"]
    B --> C["Map without guessing"]
    C --> D["Select current versions"]
    D --> E["Apply current access"]
    E --> F["Trace updates and deletion"]
    F --> G["Per-source readiness decision"]
```

## Hands-On Check

Load `RIV-FDE-1.0.0` and confirm six sources and fourteen records. Keep unresolved access freshness and purpose gaps open for the identity chapter.

## Decision

This notebook creates data-readiness teaching artifacts from synthetic fixtures. It writes no production artifact and provides no customer or Databricks validation.

## Takeaway

Do not make text searchable until its authority, current state, and deletion path are reviewable.

## 0 - The Challenge

## Situation

A payload-presence check accepts all fourteen Riverside records. That would leave an old policy searchable, scramble PDF meaning, widen rights from a missing value, turn a renamed field into a silent blank, preserve a deleted autosave, and authorize a disabled contractor from stale groups.

## Sketch

```mermaid
flowchart LR
    A["6 sources and 14 records"] --> B["Naive flatten and upsert"]
    B --> C["Stale, malformed, broad, or undeletable content"]
    C --> D["Source contracts and held-for-review paths"]
    D --> E["Lifecycle and current-access checks"]
    E --> F["Per-source readiness verdict"]
```

## Hands-On Check

Run the local checks and record which failure path each seeded record reaches. Retain the environment, commit, fixture version, method, result, and limitations before citing an observation.

## Decision

Use a per-source decision backed by distinct lifecycle, parser, contract, deletion, and access checks. Do not replace those results with one aggregate pass rate.

## Takeaway

The local run can prove that the fixture rules detect seeded failures. It cannot prove customer data or production readiness.

In [ ]:
# -- Load and validate the frozen case ---------------------------------------
from collections import Counter, defaultdict
from copy import deepcopy
from pathlib import Path
import json

from jsonschema import Draft202012Validator

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "AUTHORING_GUIDE.md").is_file() and (candidate / "learning" / "role-based-tracks" / "fde" / "shared").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the ai-portfolio repository.")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
SHARED_DIR = REPO_ROOT / "learning" / "role-based-tracks" / "fde" / "shared"
FIXTURE_DIR = SHARED_DIR / "fixtures"
SCHEMA_DIR = SHARED_DIR / "schemas"

def load_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))

engagement = load_json(FIXTURE_DIR / "riverside-engagement-v1.json")
samples = load_json(FIXTURE_DIR / "riverside-source-samples-v1.json")
facts = load_json(FIXTURE_DIR / "expected-facts-v1.json")
for name, value, schema_name in (
    ("engagement", engagement, "riverside-engagement.schema.json"),
    ("samples", samples, "riverside-source-samples.schema.json"),
    ("facts", facts, "expected-facts.schema.json"),
):
    errors = sorted(
        Draft202012Validator(load_json(SCHEMA_DIR / schema_name)).iter_errors(value),
        key=lambda error: list(error.path),
    )
    assert not errors, f"{name}: {errors[0].message if errors else 'schema error'}"

records = samples["records"]
print(f"Case: {engagement['fixture_id']} | version: {engagement['fixture_version']}")
print(f"Sources: {len(engagement['source_inventory'])} | sample records: {len(records)}")
print(f"Expected facts: {len(facts['facts'])}")

## First Check - Why Payload Presence Fails

## Situation

A record with a document ID and payload can still be stale, unreadable, unauthorized, or deleted.

## Sketch

A naive check sees an ID and payload, marks the record ready, and lets unsafe content enter search.

## Hands-On Check

Predict whether the naive rule accepts all fourteen records. Then run the next cell and compare that count with the candidates left after source-specific controls.

## Decision

Treat presence as a transport check, not a readiness decision.

## Takeaway

The result should expose why every later gate participates in readiness.

In [ ]:
# -- Expose the naive readiness failure --------------------------------------
naive_ready = [record for record in records if record.get('document_id') and record.get('payload')]
non_index_decisions = {
    'exclude_from_current_policy_index', 'quarantine_until_duplicate_and_ocr_review',
    'exclude_and_emit_tombstone', 'quarantine_for_owner_resolution',
    'quarantine_schema_drift', 'preserve_for_incident_replay',
    'allow_with_request_purpose', 'deny_and_reconcile',
}
controlled_candidates = [record for record in records if record['expected_onboarding']['decision'] not in non_index_decisions]
print(f'[Measured - local fixture] Naive ready count: {len(naive_ready)} of {len(records)}')
print(f'[Measured - local fixture] Candidates before deeper gates: {len(controlled_candidates)}')
print('Prediction resolution: A is the naive result; B is the required control model.')

## 1 - Inventory the Source and Its Owner

## Situation

A source row is useful only when Riverside names who can approve its purpose, access, retention, refresh, and deletion. Estimated counts and freshness targets remain customer claims until their owners validate them.

## Sketch

```mermaid
flowchart LR
    A["Frozen source inventory"] --> B["Join records by stable source ID"]
    B --> C{"Owner, purpose, access, refresh, and deletion known?"}
    C -->|"No"| D["Block and assign an owner"]
    C -->|"Yes"| E["Sample the approved scope"]
    E --> F["DATA-01"]
```

## Hands-On Check

Change the required samples per source from one to three. Identify the sources that become blocked and state whether three is measured evidence or an invented policy choice.

## Decision

- **Avoid:** treating estimates as observations or joining by display name.
- **Instead:** preserve evidence labels and join by stable IDs so aliases and reordering cannot attach the wrong authority.

## Takeaway

Inventory is an ownership decision, not a row count.

In [ ]:
# -- Build and check DATA-01 -------------------------------------------------
inventory = {source['source_id']: source for source in engagement['source_inventory']}
records_by_source = defaultdict(list)
for record in records:
    records_by_source[record['source_id']].append(record)
orphan_ids = sorted(record['record_id'] for record in records if record['source_id'] not in inventory)
unsampled_ids = sorted(set(inventory) - set(records_by_source))
source_types = sorted({source['source_type'] for source in inventory.values()})
required_samples_per_source = 1  # CHANGE THIS: try 3, then justify it.
sample_shortfalls = {source_id: required_samples_per_source - len(records_by_source[source_id]) for source_id in inventory if len(records_by_source[source_id]) < required_samples_per_source}
assert samples['fixture_version'] == engagement['fixture_version'] == 'RIV-FDE-1.0.0'
assert not orphan_ids and not unsampled_ids
assert set(source_types) == {'API', 'ERP', 'PDF', 'text'}
assert all(source['owner_person_id'] for source in inventory.values())
print(f'[Measured - local fixture] Sources joined: {len(inventory)}; types: {source_types}')
print(f'[Measured - local fixture] Orphans: {orphan_ids}; unsampled: {unsampled_ids}')
print(f'Policy result at sample threshold {required_samples_per_source}: {sample_shortfalls}')
print('[Customer claim retained] Counts, freshness, ownership, and deletion behavior remain unvalidated.')

## 2 - Map, Parse, or Hold for Review

## Situation

Flattening every payload can break column meaning, hide unreadable text, turn a missing territory into an unsupported rights grant, and convert a renamed required field into a silent blank.

## Sketch

```mermaid
flowchart LR
    A["PDF, text, ERP, and API"] --> B["Source-specific adapter"]
    B --> C{"Contract and quality pass?"}
    C -->|"No"| D["Hold for review with a safe reason"]
    C -->|"Deleted"| E["Emit deletion marker"]
    C -->|"Yes"| F["Create versioned parsed document"]
    D --> G["Owner review and replay"]
```

## Hands-On Check

Run the next cell and inspect the separate paths for stale policy, columns, duplicate pages, low-confidence text, deletion, missing territory, and required-field drift.

## Decision

Never default missing territory to worldwide and never log raw sensitive text in an operational review record. Store a safe reason plus lineage and replay from governed storage.

## Takeaway

Uncertainty stays non-searchable until its owner resolves it.

In [ ]:
# -- Build DATA-02 mappings and DATA-03 quality evidence ---------------------
def repeated_pdf_pages(record: dict) -> list[int]:
    if record['payload_type'] != 'pdf_pages':
        return []
    seen, duplicates = set(), []
    for page in record['payload']['pages']:
        fingerprint = json.dumps(page.get('text_blocks', []), sort_keys=True)
        if fingerprint in seen:
            duplicates.append(page['page_number'])
        seen.add(fingerprint)
    return duplicates

def local_disposition(record: dict, ocr_threshold: float = 0.80) -> tuple[str, list[str]]:
    reasons, payload = [], record['payload']
    if record['lifecycle']['status'] == 'superseded': reasons.append('not_current')
    if 'deleted' in record['deletion_state']: reasons.append('tombstone_required')
    if record['payload_type'] == 'pdf_columns': reasons.append('layout_parser_required')
    duplicates = repeated_pdf_pages(record)
    if duplicates: reasons.append(f'duplicate_pages:{duplicates}')
    if record['payload_type'] == 'pdf_pages':
        low_ocr = [page['page_number'] for page in payload['pages'] if page.get('ocr_confidence', 1.0) < ocr_threshold]
        if low_ocr: reasons.append(f'low_ocr_pages:{low_ocr}')
    if record['payload_type'] == 'erp_row' and 'territory' in payload and payload['territory'] is None: reasons.append('territory_unknown')
    if record['payload_type'] == 'api_page' and any('status' not in item for item in payload['items']): reasons.append('required_status_schema_drift')
    if any(reason.startswith(('duplicate_pages', 'low_ocr_pages')) for reason in reasons): return 'QUARANTINE_PARSE', reasons
    if 'territory_unknown' in reasons or 'required_status_schema_drift' in reasons: return 'QUARANTINE_CONTRACT', reasons
    if 'tombstone_required' in reasons: return 'TOMBSTONE', reasons
    if 'not_current' in reasons: return 'EXCLUDE_VERSION', reasons
    if 'layout_parser_required' in reasons: return 'CONDITIONAL_PARSE', reasons
    return 'ACCEPT_MAPPING', reasons

mapping_results = {record['record_id']: local_disposition(record) for record in records}
disposition_counts = Counter(result[0] for result in mapping_results.values())
quality_report = {
    'artifact_id': 'DATA-03',
    'claim_class': 'Measured - local fixture',
    'fixture_version': samples['fixture_version'],
    'sample_size': len(records),
    'disposition_counts': dict(sorted(disposition_counts.items())),
    'quarantine_record_ids': sorted(
        record_id for record_id, result in mapping_results.items()
        if result[0] in {'QUARANTINE_PARSE', 'QUARANTINE_CONTRACT'}
    ),
    'limitations': [
        'Synthetic seeded-shape checks are not representative parser benchmarks.',
        'No customer source, Databricks job, or vector index was tested.',
    ],
}
assert mapping_results['REC-RIV-PDF-001'][0] == 'EXCLUDE_VERSION'
assert mapping_results['REC-RIV-PDF-003'][0] == 'CONDITIONAL_PARSE'
assert mapping_results['REC-RIV-PDF-004'][0] == 'QUARANTINE_PARSE'
assert mapping_results['REC-RIV-TEXT-002'][0] == 'TOMBSTONE'
assert mapping_results['REC-RIV-ERP-002'][0] == 'QUARANTINE_CONTRACT'
assert mapping_results['REC-RIV-API-002'][0] == 'QUARANTINE_CONTRACT'
print(f"[Measured - local fixture] DATA-03 sample size: {quality_report['sample_size']}")
print(f"[Measured - local fixture] Dispositions: {quality_report['disposition_counts']}")
print(f"[Measured - local fixture] Quarantine: {quality_report['quarantine_record_ids']}")
print('Prediction resolution: every shortcut violates a different contract boundary.')
print('Limitation: seeded-shape detection is not a parser benchmark.')

### Walk Through the Record Decision

## Situation

One Riverside record can fail for several reasons, so the order of checks matters.

## Sketch

Check current or deleted status first, then layout, repeated pages, readability, and required business fields before deciding what happens to the record.

## Hands-On Check

Trace one stale policy, the repeated-page rights PDF, the deleted autosave, the missing-territory ERP row, and the renamed API field through `local_disposition`.

## Decision

Check lifecycle before parse quality; preserve layout meaning; keep duplicate and readability checks separate; never invent a business default; require a versioned decision for field aliases.

## Takeaway

The code is a visible teaching model. Production ingestion remains owned by the linked remote pipeline, not by this notebook.

## 3 - Preserve Identity Before Detecting Duplicates

## Situation

The deleted chapter 38 autosave resembles the current chapter. Merging them because the text looks alike could resurrect deleted content or discard the current version. Similar text does not prove shared owner, access, lifecycle, or identity.

## Sketch

```mermaid
flowchart LR
    A["Tenant and canonical source location"] --> B["Stable document ID"]
    B --> C["Source version and content fingerprint"]
    C --> D{"Current and active?"}
    D -->|"No"| E["Keep history; exclude or delete downstream copies"]
    D -->|"Yes"| F["Current searchable view"]
```

## Hands-On Check

Set `include_superseded` to true. The assertion should fail because the 2024 policy returns. Confirm that the deleted autosave stays absent while the current chapter remains.

## Decision

Scope identity and versions before comparing content. Keep immutable versions plus a current view rather than overwriting history.

## Takeaway

Similarity can flag a review; identity and lifecycle decide what is current.

In [ ]:
# -- Measure and check version selection ------------------------------------
def stable_version_key(record: dict) -> tuple[str, str, str, str]:
    return record['tenant_id'], record['document_id'], record['source_version'], record['content_hash']
naive_policy_ids = [record['document_id'] for record in records if record['source_id'] == 'SRC-PDF-POLICY-001']
include_superseded = False  # CHANGE THIS: True should fail the current-view check.
selected_policy_ids = [record['document_id'] for record in records if record['source_id'] == 'SRC-PDF-POLICY-001' and (include_superseded or record['lifecycle']['status'] == 'current') and record['deletion_state'] == 'active']
current_ids = [record['document_id'] for record in records if record['lifecycle']['status'] == 'current' and record['deletion_state'] == 'active']
version_keys = [stable_version_key(record) for record in records]
assert 'DOC-POL-AI-2024' in naive_policy_ids and 'DOC-POL-AI-2024' not in selected_policy_ids
assert 'DOC-POL-AI-2026' in selected_policy_ids
assert 'DOC-MANUSCRIPT-ARIA-038-AUTOSAVE' not in current_ids
assert len(version_keys) == len(set(version_keys))
print(f'[Measured - local fixture] Naive policies: {naive_policy_ids}')
print(f'[Measured - local fixture] Current policies: {selected_policy_ids}')
print('PASS: lifecycle selection excludes stale and deleted records without erasing lineage.')

## 4 - Follow Every Page and Stop on Undocumented Shape Changes

## Situation

The first workflow API page returns 100 records and a cursor. The second returns 37 more but renames `status` without a contract version. Reading one page loses records; permissive parsing turns the renamed field into a blank.

## Sketch

```mermaid
flowchart LR
    A["Page 1: 100 records"] --> B{"More pages?"}
    B -->|"Yes"| C["Page 2: 37 records"]
    C --> D{"Required fields match the approved contract?"}
    D -->|"No"| E["Hold page; keep previous sync checkpoint"]
    D -->|"Yes"| F["Replay-safe merge"]
    F --> G["Advance sync checkpoint"]
```

## Hands-On Check

Predict 137 records after following the cursor. Set `accept_undocumented_alias` to true and observe that a value appears, but the approved contract check still must fail.

## Decision

Advance sync state only after accepted replay-safe writes. Treat a timeout as an unknown outcome and reconcile by a stable business request key before retrying.

## Takeaway

Complete traversal and exact contracts prevent silent loss and silent blanks.

In [ ]:
# -- Measure and check pagination, drift, and idempotency -------------------
workflow_pages = sorted([record for record in records if record['payload_type'] == 'api_page'], key=lambda record: record['payload']['page'])
one_call_count = workflow_pages[0]['payload']['records_returned']
cursor_count = sum(record['payload']['records_returned'] for record in workflow_pages)
required_fields = {'task_id', 'title_id', 'status', 'assigned_user_id', 'updated_at'}
drift = []
for record in workflow_pages:
    for item in record['payload']['items']:
        missing = sorted(required_fields - set(item))
        if missing: drift.append({'record_id': record['record_id'], 'missing': missing, 'unexpected': sorted(set(item) - required_fields)})
accept_undocumented_alias = False  # CHANGE THIS only with versioned approval.
page_two_item = workflow_pages[1]['payload']['items'][0]
status = page_two_item.get('status')
if accept_undocumented_alias: status = status or page_two_item.get('workflow_status')
history = next(record for record in records if record['record_id'] == 'REC-RIV-API-003')
assert workflow_pages[0]['payload']['next_cursor'] == 'cursor-page-2'
assert cursor_count == 137 and len(drift) == 1 and status is None
assert all(attempt['idempotency_key'] is None for attempt in history['payload']['attempts'])
print(f'[Measured - local fixture] One call: {one_call_count}; cursor complete: {cursor_count}; truncation: {(cursor_count-one_call_count)/cursor_count:.1%}')
print(f'[Measured - local fixture] Drift: {drift}')
print('Prediction resolution: traversal finds 137; required-field drift blocks page two.')
print('Limitation: this does not prove API replay, cursor expiry, or Delta merge behavior.')

## 5 - Use Current Identity and Fail Closed

## Situation

Riverside's disabled contractor still appears in an old `ROLE-EDITOR` group. Filtering only on copied roles would grant access after the contract ended. Authorization also needs tenant, region, title, record access, and request purpose.

## Sketch

```mermaid
flowchart LR
    A["Current request context"] --> B{"Identity enabled now?"}
    B -->|"No"| C["Deny and reconcile"]
    B -->|"Yes"| D{"Tenant, region, title, role, and purpose match?"}
    D -->|"No"| C
    D -->|"Yes"| E["Authorized candidate"]
    E --> F["Audit decision without content text"]
```

## Hands-On Check

Predict whether the stale role check allows the contractor, then compare it with current-context authorization. Also test the editor against a manuscript, a rights schedule, and an unrelated purpose.

## Decision

Intersect current identity and request context with each record's access rules. Tenant membership alone does not imply purpose, and stale copied groups never override a disabled identity.

## Takeaway

Every retrieval candidate needs current authority, not yesterday's snapshot.

In [ ]:
# -- Measure and check ACL decisions ----------------------------------------
identity_records = {record['document_id']: record for record in records if record['payload_type'] == 'identity_record'}
manuscript = next(record for record in records if record['document_id'] == 'DOC-MANUSCRIPT-ARIA-037')
rights = next(record for record in records if record['document_id'] == 'DOC-RIGHTS-ARIA-001')
def acl_role_scope(entry: str) -> tuple[str, str | None]:
    role, separator, scope = entry.partition(':')
    return role, scope if separator else None
def authorize(identity_record: dict, record: dict, purpose: str) -> tuple[bool, str]:
    identity = identity_record['payload']
    if not identity.get('enabled', False): return False, 'identity_disabled'
    if record['tenant_id'] not in identity.get('tenant_ids', []): return False, 'tenant_mismatch'
    if record['region'] != identity.get('region_id'): return False, 'region_mismatch'
    title_id = record['payload'].get('title_id')
    if title_id and title_id not in identity.get('title_ids', []): return False, 'title_mismatch'
    if purpose not in {'editorial_retrieval', 'rights_review'}: return False, 'purpose_not_allowed'
    roles = set(identity.get('role_ids', identity.get('direct_role_ids', [])))
    if any(role in roles and (scope is None or scope == title_id) for role, scope in map(acl_role_scope, record['acl'])): return True, 'acl_match'
    return False, 'acl_no_match'
editor, contractor = identity_records['API-USER-EDITOR-017'], identity_records['API-USER-CONTRACTOR-044']
stale_roles = set(contractor['payload']['stale_nested_group_role_ids'])
naive_allowed = any(acl_role_scope(entry)[0] in stale_roles for entry in manuscript['acl'])
decisions = {
    'editor_manuscript': authorize(editor, manuscript, 'editorial_retrieval'),
    'editor_rights': authorize(editor, rights, 'editorial_retrieval'),
    'contractor_manuscript': authorize(contractor, manuscript, 'editorial_retrieval'),
    'wrong_purpose': authorize(editor, manuscript, 'unrelated_analytics'),
}
assert naive_allowed is True
assert decisions['editor_manuscript'] == (True, 'acl_match')
assert decisions['editor_rights'] == (False, 'acl_no_match')
assert decisions['contractor_manuscript'] == (False, 'identity_disabled')
assert decisions['wrong_purpose'] == (False, 'purpose_not_allowed')
print(f'[Measured - local fixture] Naive contractor allowed: {naive_allowed}')
for name, decision in decisions.items(): print(f'[Measured - local fixture] {name}: {decision}')
print('Prediction resolution: stale roles allow; current identity state denies first.')
print('Limitation: this does not prove IdP or vector-filter enforcement.')

## 6 - Trace Deletion Through Every Derived Record

## Situation

Removing a source row does not prove that parsed text, chunks, vectors, and index entries disappeared. Riverside needs a path from a deletion event to every derived copy, plus evidence that the content is no longer searchable.

## Sketch

```mermaid
flowchart LR
    A["Source event or reconciliation"] --> B["Raw version"]
    B --> C["Parsed version"]
    C --> D["Versioned chunks"]
    D --> E["Vector and index records"]
    A --> F{"Delete or revoke?"}
    F -->|"Yes"| G["Deletion marker and derived deletes"]
    G --> H["Negative query or completion receipt"]
    F -->|"No"| I["Advance accepted sync checkpoint"]
```

## Hands-On Check

Set sync overlap to zero. The assertion should fail. Confirm complete lineage, replay-safe keys, reconciliation, deletion propagation, current-version survival, and target deletion evidence.

## Decision

Use overlap and reconciliation so missed or late events can repair. Collect completion evidence for every derived delete. The production implementation remains in the Databricks index operations assets.

## Takeaway

Deletion is a tracked state change with proof of downstream absence, not best-effort file removal.

In [ ]:
# -- Build and check DATA-04 lineage and deletion ---------------------------
required_lineage = {'tenant_id', 'document_id', 'source_uri', 'source_version', 'content_hash', 'acl', 'region', 'classification', 'ingested_at', 'pipeline_version', 'deletion_state'}
lineage_missing = {record['record_id']: sorted(required_lineage - set(record)) for record in records if required_lineage - set(record)}
lineage_coverage = (len(records) - len(lineage_missing)) / len(records)
initial_index = {record['document_id']: record['record_id'] for record in records if record['payload_type'] == 'text'}
index_after_sync = deepcopy(initial_index)
tombstone_ids = [record['document_id'] for record in records if 'deleted' in record['deletion_state']]
for document_id in tombstone_ids: index_after_sync.pop(document_id, None)
overlap_seconds = 300  # CHANGE THIS: 0 removes late-arrival protection.
reconciliation_enabled = True
assert lineage_coverage == 1.0
assert 'DOC-MANUSCRIPT-ARIA-038-AUTOSAVE' in initial_index and 'DOC-MANUSCRIPT-ARIA-038-AUTOSAVE' not in index_after_sync
assert 'DOC-MANUSCRIPT-ARIA-038' in index_after_sync
assert overlap_seconds > 0 and reconciliation_enabled
print(f'[Measured - local fixture] Lineage coverage: {lineage_coverage:.1%}; tombstones: {tombstone_ids}')
print(f'[Measured - local simulation] Before: {sorted(initial_index)}; after: {sorted(index_after_sync)}')
print('External validation required: target deletion, retention, and reconciliation timing.')

## 7 - Let the Weakest Material Gate Decide

## Situation

Clean records from one source can hide a serious rights, parser, or access failure in another. One average quality score would make that unsafe source look acceptable. Data readiness also says nothing yet about retrieval relevance or generated answers.

## Sketch

```mermaid
flowchart LR
    A["Source inventory"] --> B["Mapping and parsing"]
    B --> C["Quality checks"]
    C --> D["Lifecycle, access, and lineage"]
    D --> E{"Does every material gate pass?"}
    E -->|"No"| F["Blocked or excluded"]
    E -->|"Conditions remain"| G["Conditional"]
    E -->|"Yes"| H["Ready for retrieval evaluation"]
```

## Hands-On Check

Run the source verdicts. Confirm that no source becomes production-ready and identify which sources may proceed only toward bounded retrieval evaluation.

## Decision

Keep verdicts per source with blockers, evidence labels, owners, external checks, and expiry triggers. Say "ready for retrieval evaluation," never "RAG ready."

## Takeaway

The weakest safety-relevant gate controls exposure.

In [ ]:
# -- Build and check DATA-05 retrieval readiness ----------------------------
blocking = {'QUARANTINE_PARSE', 'QUARANTINE_CONTRACT'}
conditional = {'CONDITIONAL_PARSE', 'TOMBSTONE', 'EXCLUDE_VERSION'}
source_verdicts = {}
for source_id, source_records in records_by_source.items():
    dispositions = {mapping_results[record['record_id']][0] for record in source_records}
    if source_id == 'SRC-API-IDENTITY-001':
        verdict, rationale = 'BLOCKED', 'Disabled identity and stale nested group require reconciliation.'
    elif dispositions & blocking:
        verdict, rationale = 'BLOCKED', 'At least one sample is in parser or contract quarantine.'
    elif dispositions & conditional:
        verdict, rationale = 'CONDITIONAL', 'Lifecycle, parser, or deletion controls need owner evidence.'
    else:
        verdict, rationale = 'CONDITIONAL', 'Local mapping passes; authority and target enforcement are unvalidated.'
    source_verdicts[source_id] = {'verdict': verdict, 'rationale': rationale, 'external_validation_required': True}
overall_verdict = 'BLOCKED' if any(result['verdict'] == 'BLOCKED' for result in source_verdicts.values()) else 'CONDITIONAL'
assert overall_verdict == 'BLOCKED'
for source_id in ('SRC-PDF-RIGHTS-001', 'SRC-ERP-CATALOG-001', 'SRC-API-WORKFLOW-001', 'SRC-API-IDENTITY-001'): assert source_verdicts[source_id]['verdict'] == 'BLOCKED'
for source_id in ('SRC-PDF-POLICY-001', 'SRC-TEXT-MANUSCRIPT-001'): assert source_verdicts[source_id]['verdict'] == 'CONDITIONAL'
assert all(result['external_validation_required'] for result in source_verdicts.values())
print(f'[Measured - local fixture] Overall DATA-05 verdict: {overall_verdict}')
for source_id, result in source_verdicts.items(): print(f"  {source_id}: {result['verdict']} - {result['rationale']}")
print('Prediction resolution: no production claim exists; every source is conditional or blocked.')
print('Next gate: retrieval and citation evaluation for approved current views only.')

## 8 - Handoff to Identity, Databricks, and Retrieval Evaluation

## Situation

The local notebook can show how seeded records should be classified, but Riverside still needs target-system proof for identity freshness, storage and index controls, deletion, scale, region, and operations.

## Sketch

```mermaid
flowchart LR
    A["Local fixture checks"] --> B["DATA-01 through DATA-05"]
    B --> C["Authorized run record"]
    C --> D["Databricks validation"]
    C --> E["Identity validation"]
    D --> F["Retrieval and citation evaluation"]
    E --> F
    F --> G["Customer readiness decision"]
```

## Hands-On Check

Before handoff, confirm that every named technique has one of three states: executable local check, explained link to the owning implementation, or named external evidence requirement. Populate the quality report and run record only after an authorized run preserves environment, commit, fixture version, method, observations, and limitations.

## Decision

Carry identity freshness, purpose, and title scope into the identity chapter. Carry only approved current views into hybrid-search and RAG evaluation. Keep OCR benchmarks, legal retention, cloud access, networking, scale, cost, region, and target deletion as externally owned checks.

## Takeaway

Local data checks open the next evaluation gate; they do not close the customer-readiness decision.